## Lab 5 - Part 1: Automated Data Ingestion & Versioning
-   **Course:** Engineering of Intelligent Models
-   **Module:** M3. Model Orchestration & Automation
-   **Focus:** Dynamic API Ingestion, Hydra Configuration, and DVC Versioning
-   **Branch:** `Lab5`

### 1\. Goal of the Laboratory
The objective of this first segment is to engineer a robust, reproducible data ingestion pipeline. In a production environment, data is never static. We require a mechanism to pull historical weather data dynamically without altering the underlying Python code.

By the end of this notebook, you will have:
1.  Designed a hierarchical Hydra configuration to manage API endpoints, predefined geographical locations, and meteorological variables.
2.  Implemented a Python module utilizing the official Open-Meteo client to retrieve historical data efficiently.
3.  Versioned the resulting raw dataset using DVC to ensure cryptographic lineage for future model training.

### 2. Version Control: Switching Branches
Before proceding with this Lab, ensure your repository is clean and branched.

Switch to a new branch for Lab 5.

In [ ]:
# Commit your Lab 4 progress
!git add .
!git commit -m "Complete Lab 4: Apache Airflow initial orchestration."

# Create and switch to the Lab4 branch
!git checkout -b Lab5

### 3\. Configuration Management (Hydra)

Hardcoding coordinates, dates, or variable names into your ingestion script creates "Configuration Debt". We will extract these parameters into a structured YAML file. This allows us to switch our data context (e.g., from Lisbon to Porto) purely via command-line overrides during Airflow orchestration.

Refactor the configuration file at `conf/api/openmeteo.yaml` to support new configurations.

```yaml
api:
  endpoint: "https://archive-api.open-meteo.com/v1/archive"
  timezone: "auto"

# Predefined locations dictionary. Allows easy swapping via Hydra
locations:
  Sintra:
    latitude: 38.801
    longitude: -9.3783
  Porto:
    latitude: 41.1496
    longitude: -8.6110
  Lisbon:
    latitude: 38.7167
    longitude: -9.1333

# Default location selected for ingestion
target_location: "Sintra"

# Data ranges
date_range:
  start_date: "2026-02-06"
  end_date: "2026-02-20"

# Target variables as per Open-Meteo Historical API documentation
variables:
  - "temperature_2m"
  - "relative_humidity_2m"
  - "precipitation"

output:
  raw_data_path: "data/raw/historical_weather.csv"
```

### 4\. The Ingestion Implementation

We will use the `openmeteo-requests`, `requests-cache`, and `retry-requests` libraries (already included in your `requirements.txt` from Lab 4) to ensure our API calls are resilient to network failures and rate limits.

Create the a new ingestion script at `src/ingestion/get_historical_data.py` (we're preserving the older `get_data.py` for now).

```python
import hydra
from omegaconf import DictConfig
from typing import Dict, Any
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import logging
import os

# Configure basic logging
logging.basicConfig(level=logging.INFO, format='[%(asctime)s] %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


@hydra.main(version_base=None, config_path="../../conf", config_name="config")
def ingest_data(cfg: DictConfig) -> None:
    """
    Retrieves historical weather data from Open-Meteo API based on Hydra configuration
    and saves it to the specified raw data path.
    """
    logger.info("--- Starting Data Ingestion Pipeline ---")

    # 1. Setup the Open-Meteo API client with cache and retry on error
    cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
    retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
    openmeteo = openmeteo_requests.Client(session=retry_session)

    # 2. Extract configurations
    location_name = cfg.ingestion.historical_weather.target_location
    coords = cfg.ingestion.historical_weather.locations[location_name]
    dates = cfg.ingestion.historical_weather.date_range
    variables = cfg.ingestion.historical_weather.variables
    output_path = cfg.ingestion.historical_weather.output.raw_data_path

    logger.info(f"Target Location: {location_name} (Lat: {coords.latitude}, Lon: {coords.longitude})")
    logger.info(f"Date Range: {dates.start_date} to {dates.end_date}")

    # 3. Construct the API Payload
    params = {
        "latitude": coords.latitude,
        "longitude": coords.longitude,
        "start_date": dates.start_date,
        "end_date": dates.end_date,
        "hourly": list(variables),
        "timezone": cfg.ingestion.openmeteo.api.timezone
    }

    # 4. Execute the API Request
    logger.info("Fetching data from Open-Meteo Historical API...")
    url = cfg.ingestion.historical_weather.api.endpoint
    responses = openmeteo.weather_api(url, params=params)

    # Process the first location (we only requested one)
    response = responses[0]
    hourly = response.Hourly()

    # 5. Process Data into a Pandas DataFrame
    # Note: Constructing the time index correctly to match the array lengths
    start_time = pd.to_datetime(hourly.Time(), unit="s", utc=True)
    end_time = pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True)
    interval = pd.Timedelta(seconds=hourly.Interval())

    # Create the date range - this ensures the index length matches the variables
    date_range = pd.date_range(
        start=start_time,
        end=end_time,
        freq=interval,
        inclusive="left"
    )

    # Initialize the dictionary with the date and the location name
    hourly_data: Dict[str, Any] = {
        "date": date_range,
        "location": [location_name] * len(date_range)  # Broadcast location name to all rows
    }

    # Dynamically map the requested variables to the response arrays
    for idx, var_name in enumerate(variables):
        hourly_data[var_name] = hourly.Variables(idx).ValuesAsNumpy()

    df = pd.DataFrame(data=hourly_data)

    # Convert date to a more readable format for the CSV
    df['date'] = df['date'].dt.strftime('%Y-%m-%d %H:%M:%S')

    # 6. Ensure output directory exists and save
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)

    logger.info(f"Successfully ingested {len(df)} records.")
    logger.info(f"Data saved to {output_path}")
    logger.info("--- Data Ingestion Complete ---")


if __name__ == "__main__":
    ingest_data()
```

### 5\. Data Lineage and Versioning (DVC)

Once the script is executed and the CSV is generated, we must freeze this specific state of the data. Machine Learning reproducibility dictates that we must always know exactly which data was used to train a specific model.

##### Step 1: Execute the script locally to generate the initial dataset.

In [1]:
# Change the root directory for the script to work (if you're running my repo dir structure)
import os
os.chdir("../../")

!python src/ingestion/get_historical_data.py

##### Step 2: Make sure that `raw.dvc` is on git to allow DVC folder tracking.

In [7]:
# 1. Tell DVC to update its snapshot of the entire folder
!dvc add data/raw

# 2. DVC will automatically update the data/raw.dvc file with a new hash.
# Now, you just commit that updated pointer file to Git:
!git add data/raw.dvc


To track the changes with git, run:

	git add 'data\raw.dvc'

To enable auto staging, run:

	dvc config core.autostage true


⠋ Checking graph



##### Step 3: Commit the DVC tracking metadata to Git. Do not commit the raw CSV file to Git.

In [4]:
!git add data/raw/historical_weather.csv.dvc
#!git commit -m "feat: Ingest historical weather data for Lisbon (2015-2023) and add to DVC"

fatal: pathspec 'data/raw/historical_weather.csv.dvc' did not match any files
